# Raymond DPO 对齐训练

**前置条件：**
- 已完成 SFT 训练（`raymond_train.ipynb`），merged_model 已保存到 Google Drive
- 已运行 `generate_preference_data.py` + `clean_preference_data.py`，偏好数据保存到 Drive

**DPO vs PPO 选哪个？**

| | DPO | PPO (RLHF) |
|---|---|---|
| 难度 | 简单，一步完成 | 复杂，需要先训 RM |
| 稳定性 | 非常稳定 | 可能不稳定 |
| 效果 | 对话风格对齐很好 | 理论上更强，但调参难 |
| 适用 | **推荐先试这个** | 对 DPO 效果不满意再试 |

**DPO 核心公式：**
$$\mathcal{L}_{\text{DPO}} = -\mathbb{E}\left[\log \sigma\left(\beta \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)\right]$$

- $\pi_\theta$：正在训练的 Policy 模型
- $\pi_{\text{ref}}$：冻结的 Reference 模型（SFT checkpoint，防止模型跑偏）
- $y_w$ / $y_l$：chosen / rejected 回复
- $\beta$：KL 散度惩罚系数，控制偏离 ref 的程度（默认 0.1）

| 参数 | T4 | A100 | H100 |
|---|---|---|---|
| beta | 0.1 | 0.1 | 0.1 |
| batch_size | 1 | 2 | 4 |
| learning_rate | 5e-6 | 5e-6 | 2e-6 |
| 预计时长 | ~40分钟 | ~15分钟 | ~6分钟 |

## Cell 1：安装依赖

In [ ]:
!pip install -q llamafactory
!llamafactory-cli version

## Cell 2：挂载 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json

# 偏好数据路径（从本地上传到 Drive）
PREF_DATA_PATH = '/content/drive/MyDrive/raymond/raymond_preference.json'
# SFT merged model 路径（DPO 从这里开始）
SFT_MODEL_PATH = '/content/drive/MyDrive/raymond/merged_model'

assert os.path.exists(PREF_DATA_PATH), f'找不到偏好数据: {PREF_DATA_PATH}'
assert os.path.exists(SFT_MODEL_PATH), f'找不到 SFT 模型: {SFT_MODEL_PATH}'

with open(PREF_DATA_PATH) as f:
    pref_data = json.load(f)
print(f'偏好训练样本数: {len(pref_data)} 条')

## Cell 3：准备数据目录

In [ ]:
import os, json, shutil

DATA_DIR = '/content/llama_factory_data'
os.makedirs(DATA_DIR, exist_ok=True)

shutil.copy(PREF_DATA_PATH, f'{DATA_DIR}/raymond_preference.json')

# DPO 数据集注册
# LLaMA-Factory DPO 格式：conversations + chosen + rejected
dataset_info = {
    'raymond_pref': {
        'file_name': 'raymond_preference.json',
        'formatting': 'sharegpt',
        'columns': {
            'messages': 'conversations',
            'chosen': 'chosen',
            'rejected': 'rejected'
        },
        'tags': {
            'role_tag': 'from',
            'content_tag': 'value',
            'user_tag': 'human',
            'assistant_tag': 'gpt',
            'system_tag': 'system'
        }
    }
}
with open(f'{DATA_DIR}/dataset_info.json', 'w') as f:
    json.dump(dataset_info, f, indent=2)

print('数据目录准备完成:', os.listdir(DATA_DIR))

## Cell 4：检查 GPU

In [ ]:
!nvidia-smi
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU: {name} | 显存: {mem:.1f} GB')
    if mem >= 70:
        print('→ H100 模式')
    elif mem >= 35:
        print('→ A100 模式')
    else:
        print('→ T4 模式')

## Cell 5：生成 DPO 训练配置

关键超参说明：
- `pref_beta`（β）：KL 惩罚强度。越大 = 越不敢偏离 SFT；越小 = 对齐更激进但可能不稳定。通常 0.05~0.2
- `pref_loss`：损失函数类型。`sigmoid`（原始 DPO）/ `ipo`（更稳定）/ `orpo`（无需 ref model）
- learning_rate：DPO 要比 SFT 小 10x，否则容易破坏 SFT 能力

In [ ]:
import torch, yaml, os

DPO_OUTPUT_DIR = '/content/drive/MyDrive/raymond/dpo_output'
os.makedirs(DPO_OUTPUT_DIR, exist_ok=True)

mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

if mem_gb >= 70:  # H100
    quant_config = {}
    batch_size = 4
    grad_accum = 2
    learning_rate = 2e-6
elif mem_gb >= 35:  # A100
    quant_config = {}
    batch_size = 2
    grad_accum = 4
    learning_rate = 5e-6
else:  # T4
    quant_config = {
        'quantization_bit': 4,
        'quantization_method': 'bnb',
    }
    batch_size = 1
    grad_accum = 8
    learning_rate = 5e-6

dpo_config = {
    # 模型：从 SFT merged model 开始，而不是原始 base model
    'model_name_or_path': SFT_MODEL_PATH,
    'template': 'qwen3_nothink',
    'trust_remote_code': True,
    'flash_attn': 'auto',

    # 数据
    'dataset': 'raymond_pref',
    'dataset_dir': '/content/llama_factory_data',
    'cutoff_len': 2048,
    'preprocessing_num_workers': 4,

    # DPO 核心配置
    'stage': 'dpo',           # 关键：DPO 训练阶段
    'do_train': True,
    'pref_beta': 0.1,         # β 系数，KL 散度惩罚强度
    'pref_loss': 'sigmoid',   # 损失类型：sigmoid (DPO) / ipo / orpo

    # LoRA（在 SFT 适配器基础上继续训练）
    'finetuning_type': 'lora',
    'lora_rank': 16,          # DPO 用更小的 rank
    'lora_alpha': 32,
    'lora_dropout': 0.05,
    'lora_target': 'all',

    # 训练超参（比 SFT 小很多）
    'num_train_epochs': 2,
    'per_device_train_batch_size': batch_size,
    'gradient_accumulation_steps': grad_accum,
    'learning_rate': learning_rate,
    'lr_scheduler_type': 'cosine',
    'warmup_steps': 20,
    'max_grad_norm': 1.0,
    'optim': 'adamw_torch',
    'bf16': True,

    'output_dir': DPO_OUTPUT_DIR,
    'logging_steps': 10,
    'save_steps': 50,
    'plot_loss': True,
    'report_to': 'none',
    **quant_config,
}

config_path = '/content/dpo_config.yaml'
with open(config_path, 'w') as f:
    yaml.dump(dpo_config, f, default_flow_style=False, allow_unicode=True)

print(f'DPO 配置已生成')
print(f'pref_beta={dpo_config["pref_beta"]}, pref_loss={dpo_config["pref_loss"]}')
print(f'learning_rate={learning_rate}, 等效 batch={batch_size * grad_accum}')

## Cell 6：开始 DPO 训练

In [ ]:
!llamafactory-cli train /content/dpo_config.yaml

## Cell 7：验证训练结果

In [ ]:
import os, json

print('DPO 输出文件:', os.listdir(DPO_OUTPUT_DIR))

# DPO 的关键指标：rewards/chosen 和 rewards/rejected 的差（越大越好）
trainer_log = f'{DPO_OUTPUT_DIR}/trainer_log.jsonl'
if os.path.exists(trainer_log):
    logs = []
    with open(trainer_log) as f:
        for line in f:
            try:
                logs.append(json.loads(line))
            except:
                pass
    if logs:
        last = logs[-1]
        print(f'最终 loss: {last.get("loss", "N/A"):.4f}')
        # DPO 专属指标
        chosen_r = last.get('rewards/chosen', None)
        rejected_r = last.get('rewards/rejected', None)
        if chosen_r is not None and rejected_r is not None:
            margin = chosen_r - rejected_r
            print(f'reward margin (chosen - rejected): {margin:.4f}')
            print('→ margin 越大，说明模型越能区分好坏回复')
            if margin > 0.5:
                print('✓ 对齐良好')
            elif margin > 0:
                print('△ 对齐方向正确，可以继续训练')
            else:
                print('✗ margin 为负，检查数据质量或降低 beta')

## Cell 8：测试对齐效果

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

print('加载 DPO 模型...')
tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL_PATH, trust_remote_code=True)
base = AutoModelForCausalLM.from_pretrained(
    SFT_MODEL_PATH, torch_dtype=torch.bfloat16,
    device_map='auto', trust_remote_code=True
)
model = PeftModel.from_pretrained(base, DPO_OUTPUT_DIR)
model.eval()
print('加载完成')

SYSTEM = (
    '你是Raymond，在美国留学的中国研究生。你说话短而碎，每条1-15字，用\\n分隔多条消息。'
    '口头禅：66、哈、f、说白了、不好说。幽默方式是自嘲和反讽。'
)

def chat(user_input):
    messages = [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': user_input}
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors='pt', return_dict=True
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=150, temperature=0.8,
                             do_sample=True, top_p=0.9)
    prompt_len = inputs['input_ids'].shape[-1]
    return tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True)

# 测试：这些问题以前容易触发「书面语」或「过长」回复
test_prompts = [
    '你后悔出国吗',
    '你觉得自己未来会成功吗',
    '你今天心情怎么样',
    '铲吗',
]
for q in test_prompts:
    print(f'朋友: {q}')
    print(f'Raymond (DPO): {chat(q)}')
    print('-' * 30)

## Cell 9（训练完成后）：合并并保存 DPO 模型

合并后替换 Ollama 中的 GGUF 模型，即可享受对齐效果。

In [ ]:
DPO_MERGED_DIR = '/content/drive/MyDrive/raymond/dpo_merged_model'
os.makedirs(DPO_MERGED_DIR, exist_ok=True)

print('合并 DPO LoRA adapter...')
merged = model.merge_and_unload()
merged.save_pretrained(DPO_MERGED_DIR)
tokenizer.save_pretrained(DPO_MERGED_DIR)
print(f'合并完成: {DPO_MERGED_DIR}')
print('下一步：用 llama.cpp 量化为 GGUF，替换 Ollama 中的模型文件')